In [6]:
# ============================================================================
# CELL 1: Imports from existing stable modules (unchanged)
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import astropy.units as u
from astropy.cosmology import Planck18 as cosmo
from astropy.coordinates import SkyCoord
from dustmaps.sfd import SFDQuery
from rubin_sim.maf.slicers import UserPointsSlicer
from rubin_sim.phot_utils import DustValues
import astropy.units as u


repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
sys.path.insert(0, str(repo_root / "py_files"))

# Import from YOUR existing modules (these are NOT modified)

from slsn_metrics.constants import z_from_comoving_fast, dm_from_z
from slsn_metrics.model import LC, CatalogInputs, atomic_save_pickle
from slsn_metrics.metrics import SLSN_Detect_Metric
from slsn_metrics.diagnostics import (
    plot_population_diagnostics,
    characterize_template_coverage
)
from slsn_metrics.runners import (
    run_slsn_detect,
    build_filenames,
    get_distance_bounds
)

# Import the ORIGINAL population function (we'll create a new version)
from slsn_metrics.population import (
    inject_uniform_healpix,
    _cached_uniform_healpix,
    _HEALPIX_MEMO
)

print("✓ Imported all stable modules successfully")

✓ Imported all stable modules successfully


In [8]:
# ============================================================================
# CELL: Year-by-Year SLSN Analysis - ALL THREE METRICS
# ============================================================================

import numpy as np
import pandas as pd
from pathlib import Path
from slsn_metrics import (
    LC, 
    SLSN_Detect_Metric, 
    SLSN_CharacterizeMetric, 
    SLSN_SpecTriggerMetric
)
import rubin_sim.maf as maf
from rubin_sim.maf import db as mafdb
from slsn_metrics.runners import _build_detection_dataframe

# ============================================================================
# Configuration
# ============================================================================

db_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub")
output_base = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/Rubin_tests/SLSNe/yearly_all_metrics")
output_base.mkdir(parents=True, exist_ok=True)

mjd0 = 60980.5
cadences = ['baseline_v4.3.1_10yrs']
z_min_analysis = 0.1
z_max_analysis = 1.0

# Rate evolution parameters
R_ref = 1e-7
z_ref = 0.17
OH_max = 8.3

# Get template names for later
names = list(getattr(templates, "names", []))

# ============================================================================
# Storage for all years
# ============================================================================

results_by_year = []
all_year_dataframes = []  # Store DataFrames from each year

print("\n" + "="*80)
print("YEAR-BY-YEAR SLSN ANALYSIS - ALL THREE METRICS")
print("="*80)
print(f"Redshift range: {z_min_analysis} - {z_max_analysis}")
print(f"Metrics: Detection, Characterization, Spec Trigger")
print("="*80 + "\n")

# ============================================================================
# Year-by-Year Loop
# ============================================================================

for year in range(1, 11):
    
    t_start = (year - 1) * 365.25
    t_end = year * 365.25
    
    print(f"\n{'='*80}")
    print(f"YEAR {year} (Days {t_start:.0f}-{t_end:.0f})")
    print(f"{'='*80}")
    
    # ========================================================================
    # Step 1: Generate Population
    # ========================================================================
    
    print(f"  [1/5] Generating population...")
    
    pop_year = generate_SLSN_PopSlicer_with_rate_evolution(
        lc_model=templates,
        rate_model='evolving',
        R_ref=R_ref,
        z_ref=z_ref,
        OH_max=OH_max,
        z_min=0.1,
        z_max=2.0,
        t_start=t_start,
        t_end=t_end,
        gal_lat_cut=15.0,
        seed=42 + year,
        healpix_cache_file=Path("healpix_cache.pkl"),
        save_to=None,
        make_debug_plots=False
    )
    
    n_total = len(pop_year.slice_points['sid'])
    z_vals = pop_year.slice_points['z']
    z_mask = (z_vals >= z_min_analysis) & (z_vals <= z_max_analysis)
    n_in_range = int(z_mask.sum())
    
    print(f"      Generated {n_total:,} events (all z)")
    print(f"      {n_in_range:,} events in z=[{z_min_analysis}, {z_max_analysis}]")
    
    # ========================================================================
    # Step 2: Create All Three Metrics
    # ========================================================================
    
    print(f"  [2/5] Setting up metrics...")
    
    metric_detect = SLSN_Detect_Metric(
        lc_model=templates, mjd0=mjd0, store_obs_mode="meta"
    )
    metric_char = SLSN_CharacterizeMetric(
        lc_model=templates, mjd0=mjd0, store_obs_mode="meta"
    )
    metric_spec = SLSN_SpecTriggerMetric(
        lc_model=templates, mjd0=mjd0, store_obs_mode="meta"
    )
    
    # ========================================================================
    # Step 3: Run All Three Metrics
    # ========================================================================
    
    print(f"  [3/5] Running all metrics (~5-10 min)...")
    
    opsdb = str(db_dir / f"{cadences[0]}.db")
    constraint = "scheduler_note not like 'long%'"
    
    # Temporary output directory
    temp_dir = output_base / f"temp_year_{year}"
    temp_dir.mkdir(exist_ok=True)
    results_db = mafdb.ResultsDb(out_dir=str(temp_dir))
    
    # Create bundles
    bundle_detect = maf.MetricBundle(metric_detect, pop_year, constraint)
    bundle_char = maf.MetricBundle(metric_char, pop_year, constraint)
    bundle_spec = maf.MetricBundle(metric_spec, pop_year, constraint)
    
    # Run all at once
    group = maf.MetricBundleGroup(
        {
            'detect': bundle_detect,
            'characterize': bundle_char,
            'spec_trigger': bundle_spec
        },
        opsdb,
        out_dir=str(temp_dir),
        results_db=results_db
    )
    
    group.run_all()
    
    # ========================================================================
    # Step 4: Extract Results
    # ========================================================================
    
    print(f"  [4/5] Extracting results...")
    
    # Build DataFrames from obs_records
    df_detect = _build_detection_dataframe(
        dict(metric_detect.obs_records), pop_year, templates, mjd0
    )
    df_detect['detected'] = bundle_detect.metric_values.astype(bool)
    
    df_char = _build_detection_dataframe(
        dict(metric_char.obs_records), pop_year, templates, mjd0
    )
    df_char['characterized'] = bundle_char.metric_values.astype(bool)
    
    df_spec = _build_detection_dataframe(
        dict(metric_spec.obs_records), pop_year, templates, mjd0
    )
    df_spec['spec_trigger'] = bundle_spec.metric_values.astype(bool)
    
    # Merge all results
    df_year = df_detect[['sid', 'file_indx', 'z', 'distance_modulus', 
                         'detected', 'n_observations_detected', 'n_filters_detected']].copy()
    
    df_year = df_year.merge(
        df_char[['sid', 'characterized', 'n_epochs', 'n_filters_char']],
        on='sid', how='left'
    )
    
    df_year = df_year.merge(
        df_spec[['sid', 'spec_trigger', 'min_mag_near_peak']],
        on='sid', how='left'
    )
    
    # Fill NaNs
    df_year['detected'] = df_year['detected'].fillna(False).astype(bool)
    df_year['characterized'] = df_year['characterized'].fillna(False).astype(bool)
    df_year['spec_trigger'] = df_year['spec_trigger'].fillna(False).astype(bool)
    
    # Add year and event names
    df_year['year'] = year
    
    idx = pd.to_numeric(df_year['file_indx'], errors='coerce')
    df_year['event_name'] = idx.apply(
        lambda i: names[int(i)] if (pd.notna(i) and 0 <= int(i) < len(names)) else pd.NA
    )
    
    # Add tier classification
    def classify_tier(row):
        if row['spec_trigger']:
            return 'Tier 3: Spec Trigger'
        elif row['characterized']:
            return 'Tier 2: Characterized'
        elif row['detected']:
            return 'Tier 1: Detected'
        else:
            return 'Tier 0: Not Detected'
    
    df_year['tier'] = df_year.apply(classify_tier, axis=1)
    
    # Store for later
    all_year_dataframes.append(df_year)
    
    # Cleanup temp files
    import shutil
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    # ========================================================================
    # Step 5: Calculate Statistics (z-filtered)
    # ========================================================================
    
    print(f"  [5/5] Calculating statistics...")
    
    df_filtered = df_year[z_mask].copy()
    
    n_detected = int(df_filtered['detected'].sum())
    n_char = int(df_filtered['characterized'].sum())
    n_spec = int(df_filtered['spec_trigger'].sum())
    
    eff_detect = n_detected / n_in_range if n_in_range > 0 else 0.0
    eff_char = n_char / n_in_range if n_in_range > 0 else 0.0
    eff_spec = n_spec / n_in_range if n_in_range > 0 else 0.0
    
    result_row = {
        'year': year,
        't_start_days': t_start,
        't_end_days': t_end,
        'n_population': n_in_range,
        'n_detected': n_detected,
        'n_characterized': n_char,
        'n_spec_trigger': n_spec,
        'efficiency_detect': eff_detect,
        'efficiency_char': eff_char,
        'efficiency_spec': eff_spec
    }
    
    results_by_year.append(result_row)
    
    # Print year summary
    print(f"  Results:")
    print(f"      Population (z={z_min_analysis}-{z_max_analysis}): {n_in_range:>6,}")
    print(f"      Detected:                     {n_detected:>6,}  ({eff_detect:>5.1%})")
    print(f"      Characterized:                {n_char:>6,}  ({eff_char:>5.1%})")
    print(f"      Spec Trigger:                 {n_spec:>6,}  ({eff_spec:>5.1%})")

# ============================================================================
# Combine All Years
# ============================================================================

print("\n" + "="*80)
print("COMBINING RESULTS FROM ALL YEARS")
print("="*80)

df_all_years = pd.concat(all_year_dataframes, ignore_index=True)
df_results = pd.DataFrame(results_by_year)

# Save complete dataset
full_output = output_base / "all_years_all_metrics.csv"
df_all_years.to_csv(full_output, index=False)
print(f"✓ Complete dataset saved: {full_output}")

# Save summary
summary_output = output_base / "yearly_summary_all_metrics.csv"
df_results.to_csv(summary_output, index=False)
print(f"✓ Summary saved: {summary_output}")

# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "="*80)
print("AGGREGATED RESULTS: 10-YEAR SUMMARY")
print("="*80)

print("\nYear-by-Year Breakdown:")
print("-" * 80)
print(df_results[['year', 'n_population', 'n_detected', 'n_characterized', 'n_spec_trigger']].to_string(index=False))

print("\n" + "-" * 80)
print("TOTALS:")
print(f"  Total Population (10 years):  {df_results['n_population'].sum():>8,}")
print(f"  Total Detected:               {df_results['n_detected'].sum():>8,}  ({df_results['n_detected'].sum() / df_results['n_population'].sum():>6.1%})")
print(f"  Total Characterized:          {df_results['n_characterized'].sum():>8,}  ({df_results['n_characterized'].sum() / df_results['n_population'].sum():>6.1%})")
print(f"  Total Spec Trigger:           {df_results['n_spec_trigger'].sum():>8,}  ({df_results['n_spec_trigger'].sum() / df_results['n_population'].sum():>6.1%})")

print("\n" + "="*80 + "\n")

NameError: name 'templates' is not defined